<a href="https://colab.research.google.com/github/2403a52030-sketch/ML-LAB_assignment/blob/main/ML_labassignment_12_2030.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

In [3]:
df = pd.read_excel('/content/UCI_Credit_Card.csv.xlsx')
display(df.head())

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [4]:
# Remove ID column if present
if "ID" in df.columns:
    df = df.drop("ID", axis=1)

# Define features and target
X = df.drop("default.payment.next.month", axis=1)
y = df["default.payment.next.month"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
# Train Random Forest model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:,1]

# Evaluation
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_precision = precision_score(y_test, rf_pred)
rf_recall = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest Results")
print("Accuracy:", rf_accuracy)
print("Precision:", rf_precision)
print("Recall:", rf_recall)
print("ROC-AUC:", rf_auc)
print("\n")



Random Forest Results
Accuracy: 0.8116666666666666
Precision: 0.6308100929614874
Recall: 0.3579502637528259
ROC-AUC: 0.7506710534357693




In [8]:
# Train AdaBoost
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=100,
    random_state=42
)

ada_model.fit(X_train, y_train)

# Predictions
ada_pred = ada_model.predict(X_test)
ada_prob = ada_model.predict_proba(X_test)[:,1]

# Evaluation
ada_accuracy = accuracy_score(y_test, ada_pred)
ada_precision = precision_score(y_test, ada_pred)
ada_recall = recall_score(y_test, ada_pred)
ada_auc = roc_auc_score(y_test, ada_prob)

print("AdaBoost Results")
print("Accuracy:", ada_accuracy)
print("Precision:", ada_precision)
print("Recall:", ada_recall)
print("ROC-AUC:", ada_auc)

AdaBoost Results
Accuracy: 0.8171666666666667
Precision: 0.6758409785932722
Recall: 0.33308214016578747
ROC-AUC: 0.7685902161094431


In [9]:
results = []

n_estimators_list = [50, 100, 200]
max_depth_list = [None, 10, 20]

for n in n_estimators_list:
    for depth in max_depth_list:

        rf = RandomForestClassifier(
            n_estimators=n,
            max_depth=depth,
            random_state=42
        )

        rf.fit(X_train, y_train)

        pred = rf.predict(X_test)
        prob = rf.predict_proba(X_test)[:,1]

        acc = accuracy_score(y_test, pred)
        prec = precision_score(y_test, pred)
        rec = recall_score(y_test, pred)
        auc = roc_auc_score(y_test, prob)

        results.append({
            "Model": "RandomForest",
            "n_estimators": n,
            "max_depth": depth,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "ROC_AUC": auc
        })

rf_results = pd.DataFrame(results)
rf_results

,Model,n_estimators,max_depth,Accuracy,Precision,Recall,ROC_AUC
0,RandomForest,50,NaN,0.811000,0.630229,0.351922,0.747914
1,RandomForest,50,10.0,0.817667,0.666667,0.351168,0.771444
2,RandomForest,50,20.0,0.814667,0.648276,0.354182,0.759414
3,RandomForest,100,NaN,0.811667,0.630810,0.357950,0.750671
4,RandomForest,100,10.0,0.816667,0.660537,0.351922,0.772689
5,RandomForest,100,20.0,0.816833,0.657459,0.358704,0.761803
6,RandomForest,200,NaN,0.811833,0.629243,0.363225,0.754688
7,RandomForest,200,10.0,0.817167,0.665706,0.348154,0.773315
8,RandomForest,200,20.0,0.815500,0.650685,0.357950,0.765374


In [10]:
# Combine results

comparison = pd.DataFrame({
    "Model": ["Decision Tree", "AdaBoost"],
    "Accuracy": [dt_accuracy, ada_accuracy],
    "Precision": [dt_precision, ada_precision],
    "Recall": [dt_recall, ada_recall],
    "ROC_AUC": [dt_auc, ada_auc]
})

# Add Random Forest results
comparison = pd.concat([comparison, rf_results], ignore_index=True)

comparison

,Model,Accuracy,Precision,Recall,ROC_AUC,n_estimators,max_depth
0,Decision Tree,0.715167,0.370421,0.411454,0.607880,NaN,NaN
1,AdaBoost,0.817167,0.675841,0.333082,0.768590,NaN,NaN
2,RandomForest,0.811000,0.630229,0.351922,0.747914,50.0,NaN
3,RandomForest,0.817667,0.666667,0.351168,0.771444,50.0,10.0
4,RandomForest,0.814667,0.648276,0.354182,0.759414,50.0,20.0
5,RandomForest,0.811667,0.630810,0.357950,0.750671,100.0,NaN
6,RandomForest,0.816667,0.660537,0.351922,0.772689,100.0,10.0
7,RandomForest,0.816833,0.657459,0.358704,0.761803,100.0,20.0
8,RandomForest,0.811833,0.629243,0.363225,0.754688,200.0,NaN
9,RandomForest,0.817167,0.665706,0.348154,0.773315,200.0,10.0
